# Paso 2 — Entendimiento de los datos

**Proyecto:** seguimiento de residentes del Puericultorio Pérez Araníbar.

**Objetivo de este cuaderno:** entender la data que se entregó (la ficha nominal y los Excel de nutrición) e **identificar sus inconsistencias** antes de tratarlas. Aquí **no se modifica, excluye ni calcula nada que forme parte del consolidado**: solo se observa, se mide y se documenta qué problemas existen, cuántos son y dónde están.

**Qué se revisa:**

1. Archivos recibidos y su estructura.
2. Formato del texto y de las fechas en la ficha nominal (comillas, mayúsculas/minúsculas, tildes, fechas escritas de varias formas).
3. Códigos (IDs) con datos contradictorios en la ficha nominal.
4. Plantillas de los Excel de nutrición (cada mes y hogar usa columnas distintas).
5. Texto de los diagnósticos nutricionales (mayúsculas, espacios, errores de escritura).
6. Formato y consistencia de la edad.
7. Asteriscos (`*`) en los diagnósticos: cuáles corresponden a un valor que no se calculó.
8. Evaluaciones con la palabra `PROVINCIA` en lugar de mediciones.
9. IMC: valores faltantes y diferencias contra el cálculo peso/talla².
10. Evaluaciones que se pueden (o no) vincular de forma segura con la ficha nominal.
11. Cobertura del sexo, necesario para elegir la tabla de referencia INS.

**Evidencia:** cada separación de datos que se hace en este cuaderno se guarda como un Excel en `Resultados/Paso2_evidencias/`, para poder presentarla a la entidad correspondiente.

**Continuación:** el cuaderno *Paso 3 — Aplicación de tratamientos* aplica, sobre estas inconsistencias, los tratamientos y arma el consolidado.

La entrega institucional ya incluía códigos de residentes y una ficha nominal consolidada. Se confirmó el retiro previo de datos personales, pero no el detalle de los campos excluidos. Esas acciones preceden al proceso documentado aquí.

**PREGUNTA: _Falta confirmar qué campos de las fichas originales quedaron fuera de la entrega y si alguno contenía información relevante para analizar la situación legal, el PTI o sus cambios entre meses_**.

In [ ]:
# LIBRERIAS
from pathlib import Path
from collections import Counter, defaultdict
from datetime import date, datetime
import hashlib
import html
import json
import re
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.utils import column_index_from_string, get_column_letter
import unicodedata
from IPython.display import display, HTML

**Configuración.** El cuaderno se ejecuta en local: lee la ficha nominal de `Data Ficha Nominal` y los Excel mensuales de `Data Nutrición`, y escribe la evidencia en `Resultados/Paso2_evidencias`. Se calcula la huella digital (SHA-256) de cada archivo de entrada para poder demostrar, al final del proceso, que los originales no fueron alterados. `export_evidence` es la función que guarda cada separación de datos en su propio Excel.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
project_dir = Path('/content/drive/MyDrive/TA - Análisis de Datos [1INF03]')
project_dir

PosixPath('/content/drive/MyDrive/TA - Análisis de Datos [1INF03]')

In [14]:
# RUTAS DEL PROYECTO (ejecución local: el notebook vive en la carpeta "Notebooks")

nominal_dir = project_dir / "B - Data" / "A - Source"     # Excel de la ficha nominal
nutrition_dir = project_dir / "B - Data" / "A - Source"       # Excel mensuales de nutrición
evidence_dir = project_dir / "C - KDD Steps" / "B - 2: Target Dataset Selection"   # Excel de evidencia de este notebook
evidence_dir.mkdir(parents=True, exist_ok=True)

nominal_files = sorted(nominal_dir.glob("Ficha*.xlsx"))
nutrition_files = sorted(nutrition_dir.glob("Nutricion*.xlsx"))

assert len(nominal_files) == 1, "Se espera una ficha nominal consolidada."
assert len(nutrition_files) == 6, "Se esperan los seis archivos nutricionales de la entrega."

nominal_path = nominal_files[0]

# Huella digital de los archivos de entrada (permite comprobar que no se modificaron)
source_fingerprints = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in nominal_files + nutrition_files
}

pd.set_option("display.max_colwidth", 100)

# Para visualizar mejor las tablas
def show_table(table, title=None):
    if isinstance(table, pd.Series):
        table = table.to_frame()
    if title:
        display(HTML(f"<p><strong>{html.escape(title)}</strong></p>"))
    display(HTML('<div class="preparation-table">' +
                 table.to_html(index=True, escape=True, na_rep="—", border=0) + '</div>'))

# Guarda cada separación de datos en su propio Excel (evidencia para la entidad correspondiente)
def export_evidence(filename, sheets, folder=None):
    """Escribe un Excel con una hoja por tabla. `sheets` = {nombre_hoja: DataFrame}."""
    folder = folder or evidence_dir
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / filename
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            frame = frame.copy()
            # Un texto que empieza con "=" se interpretaría como fórmula en Excel: se antepone una comilla
            for column in frame.columns:
                if frame[column].dtype == "object":
                    frame[column] = frame[column].map(
                        lambda v: "'" + v if isinstance(v, str) and v.startswith("=") else v)
            frame.to_excel(writer, sheet_name=name[:31], index=False)
    print(f"Evidencia guardada: {path.relative_to(project_dir)}  ({', '.join(f'{k}: {len(v)} filas' for k, v in sheets.items())})")

# Controla lo visual
display(HTML("""<style>
.preparation-table {overflow-x:auto; margin:12px 0 22px;}
.preparation-table table {border-collapse:collapse; font-size:13px; line-height:1.5;}
.preparation-table th {background:#edf2f6; color:#24384b; padding:9px; text-align:left;}
.preparation-table td {padding:8px 9px; text-align:left; vertical-align:top; border-bottom:1px solid #e4e9ee; min-width:75px; max-width:480px; overflow-wrap:anywhere; white-space:normal;}
.preparation-table tr:nth-child(even) {background:#f8fafc;}
</style>"""))

# 1. Archivos recibidos

Se distinguen los siete archivos de entrada de los dos consolidados. La ficha nominal contiene una tabla con encabezados en la primera fila. Los libros nutricionales usan hojas por hogar, encabezados en varias filas y secciones por casa.

CARGA DE ARCHIVOS

In [15]:
# LECTURA DE LOS ARCHIVOS Y INVENTARIO

# Conservar las celdas nominales tal como llegaron, antes de preparar claves.
nominal_book = openpyxl.load_workbook(nominal_path, read_only=True, data_only=False)
nominal_sheet = nominal_book.worksheets[0]
nominal_sheet.reset_dimensions()
nominal_sheet_name = nominal_sheet.title
nominal_source_rows = list(nominal_sheet.iter_rows(values_only=True))
nominal_headers = list(nominal_source_rows[0])
nominal_book.close()

assert len(nominal_headers) == 33 # cantidad de columnas de ficha nominal
assert not any(isinstance(value, str) and value.startswith("=")
               for row in nominal_source_rows for value in row)

# En nutrición se leen tanto los resultados guardados como las fórmulas.
nutrition_sources = {}
for path in nutrition_files:
    values_book = openpyxl.load_workbook(path, data_only=True)
    formulas_book = openpyxl.load_workbook(path, data_only=False)
    sheets = []
    for sheet in values_book:
        sheets.append({
            "sheet": sheet.title,
            "rows": list(sheet.iter_rows(values_only=True)),
            "formulas": list(formulas_book[sheet.title].iter_rows(values_only=True)),
            "merged_ranges": [str(rng) for rng in sheet.merged_cells.ranges]
        })
    nutrition_sources[path.name] = sheets
    values_book.close()
    formulas_book.close()

file_inventory = [{
    "archivo": nominal_path.name, "fuente": "Nominal", "hojas": 1,
    "registros": len(nominal_source_rows) - 1,
    "organización": "Tabla nominal consolidada; encabezado en fila 1"
}]
for filename, sheets in nutrition_sources.items():
    file_inventory.append({
        "archivo": filename, "fuente": "Nutrición", "hojas": len(sheets),
        "registros": sum(bool(re.fullmatch(r"PPA_FN_\d+", str(row[1])))
                         for sheet in sheets for row in sheet["rows"] if len(row) > 1),
        "organización": "Hojas por hogar; encabezados y secciones por casa"
    })
file_inventory = pd.DataFrame(file_inventory).set_index("archivo")
show_table(file_inventory, "Inventario de la entrega")

# Evidencia: inventario de lo recibido
# export_evidence("00_inventario_archivos.xlsx", {"inventario": file_inventory.reset_index()})

,fuente,hojas,registros,organización
archivo,,,,
FichaNominal_consolidado_IDs_v2.xlsx,Nominal,1,2667,Tabla nominal consolidada; encabezado en fila 1
Nutricion_IDs_202601 - ENERO - TODOS.xlsx,Nutrición,5,141,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202602 - FEBRERO- INFANTES.xlsx,Nutrición,1,7,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Nutrición,1,6,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202604 - ABRIL - TODOS.xlsx,Nutrición,5,108,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202606 - JUNIO- TODOS.xlsx,Nutrición,5,108,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,Nutrición,2,32,Hojas por hogar; encabezados y secciones por casa


**Comentario 1:** Se recibió una ficha nominal consolidada (2 667 filas) y seis libros de nutrición con 19 hojas y 402 evaluaciones: enero (5 hojas, 141 evaluaciones), febrero (1 hoja, 7), marzo (1 hoja, 6), abril (5 hojas, 108), junio (5 hojas, 108) y agosto (2 hojas, 32).

La cobertura **no es uniforme**, ya que febrero y marzo solo evalúan a infantes, agosto solo a los dos hogares de adolescentes, y no hay libro de mayo ni de julio. Además, la ficha nominal cubre 19 meses (2025-01 a 2026-07), de modo que agosto de 2026 tiene nutrición pero no tiene ficha nominal.

In [16]:
# Convertir la ficha nominal leída con openpyxl en un DataFrame
df_nominal = pd.DataFrame(
    nominal_source_rows[1:],
    columns=nominal_headers
)
print("Dimensiones de la ficha nominal:", df_nominal.shape)
display(df_nominal.head())

Dimensiones de la ficha nominal: (2667, 33)


,periodo,IDs_FichaNominal,Sexo,Fecha de nacimiento,Departamento,Provincia,Distrito,Nacionalidad,Fecha de ingreso,Observaciones y/o comentarios de la forma de ingreso de la NNA,...,Tipo de Institución Educativa,Acceso a capacitación técnico productiva,La NNA actualmente estudia,Estado de Salud Física Actual,Presenta Discapacidad (Indicar tipo de discapacidad),Nivel Nutricional,Soporte Psico-emocional,Atenciones de salud psicológica,Atención en Salud mental especializada,Seguro de Salud (NNA)
0,"""2025_01""",PPA_FN_0001,"""F""","""2022-03-26""","""LIMA""","""LIMA""","""LIMA""","""PERUANA""","""2022-09-06""","""Presunta desprotección familiar""",...,"""Educación Básica Regular -EBR""","""No aplica""","""Si""","""Saludable""","""No tiene""","""Sobrepeso""","""Acompañamiento/Estimulación""","""NO""","""No aplica""","""SIS"""
1,"""2025_01""",PPA_FN_0002,"""M""","""2021-11-30""","""LIMA""","""LIMA""","""CALLAO""","""PERUANO""","""13/12/2021""","""Presunta desprotección familiar""",...,"""Ninguno""","""No aplica""","""No""","""Saludable""","""No tiene""","""Saludable""","""Acompañamiento/Estimulación""","""NO""","""No aplica""","""SIS"""
2,"""2025_01""",PPA_FN_0003,"""M""","""2022-03-28""","""LIMA""","""LIMA""","""CALLAO""","""PERUANO""","""2022-04-04""","""Presunta desprotección familiar""",...,"""Ninguno""","""No aplica""","""No""","""Saludable""","""No tiene""","""Riesgo de baja talla""","""Acompañamiento/Estimulación""","""NO""","""No aplica""","""SIS"""
3,"""2025_01""",PPA_FN_0004,"""M""","""2023-09-01""","""LIMA""","""LIMA""","""LIMA""","""PERUANO""","""16/01/2023""","""Presunta desprotección familiar""",...,"""Ninguno""","""No aplica""","""No""","""Saludable""","""No tiene""","""Saludable""","""Estimulación temprana""","""NO""","""No aplicca""","""SIS"""
4,"""2025_01""",PPA_FN_0005,"""F""","""2021-11-06""","""LIMA""","""LIMA""","""CALLAO""","""PERUANA""","""15/11/2021""","""Presunta desprotección familiar""",...,"""Ninguno""","""No aplica""","""No""","""Saludable""","""No tiene""","""Riesgo de baja talla""","""Acompañamiento/Estimulación""","""NO""","""No aplica""","""SIS"""


**Comentario 2:** La tabla tiene 2 667 filas × 33 columnas, con un registro por NNA y mes. Se observa desde ya que todos los textos y fechas vienen entre comillas dobles (por ejemplo `"2025_01"`, `"F"`, `"2022-03-26"`) y que el periodo usa el formato `AAAA_MM`, distinto del `AAAA-MM` de nutrición. Solo `Comunicado_DGA` es un valor lógico real (verdadero en todas las filas, es decir, sin variación). Estas comillas se deben limpiar para poder comparar con el otro archivo.

# 2. Revisión inicial y claves nominales

Los identificadores y periodos nominales incluyen comillas y el periodo usa `AAAA_MM`. Para comparar fuentes se crean claves sin comillas, con espacios uniformes y periodo `AAAA-MM`, mientras los 33 campos recibidos se conservan por separado.

Se revisarán los registros para identificar posibles inconsistencias que impidan verificar la identidad de cada NNA. El objetivo es determinar qué identificadores presentan información consistente y pueden utilizarse de manera confiable para vincular los registros de la ficha nominal con las evaluaciones de nutrición.

La validación de consistencia se realizará sobre los campos identificador del NNA, fecha de nacimiento, sexo, nacionalidad, fecha de ingreso, observaciones sobre la forma de ingreso y fecha del oficio de ingreso. El identificador y la fecha de nacimiento se consideran porque están presentes en ambas fuentes y permiten verificar que los registros correspondan al mismo NNA. El sexo y la nacionalidad se incluyen para detectar posibles contradicciones entre los registros históricos. Asimismo, se revisan la fecha de ingreso, las observaciones sobre la forma de ingreso y la fecha del oficio de ingreso, debido a que corresponden a información administrativa del ingreso que debería mantenerse consistente para un mismo evento.

Adicionalmente, se verificará la cantidad de registros y los periodos asociados a cada identificador. Dado que la ficha nominal se reporta mensualmente y el periodo analizado comprende 19 meses, se espera encontrar como máximo 19 registros por NNA, uno por cada mes. Si un identificador presenta más de 19 registros o varias fichas correspondientes al mismo periodo, se considerará una posible inconsistencia que deberá revisarse.

INCONSISTENCIAS FICHA NOMINAL

In [17]:
# PREPARACIÓN DE CLAVES NOMINALES Y REVISIÓN DE IDENTIDADES

def clean_text(value):
    # Limpia un texto: None -> "", espacios múltiples -> uno solo, quita espacios y comillas de los bordes.
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().strip('"').strip()

def serialize_original(value):
    # Convierte fechas a texto ISO (AAAA-MM-DD) para guardarlas
    return value.isoformat() if isinstance(value, (date, datetime)) else value

def parse_birth_date(value):
    # fecha de nacimiento como texto
    if isinstance(value, (date, datetime)):
        return value.strftime("%Y-%m-%d")
    for date_format in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return datetime.strptime(clean_text(value), date_format).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

def numeric_value(value):
    # retorna el valor solo si es numero
    return value if isinstance(value, (int, float)) and not isinstance(value, bool) else None

# Funciones de normalización para los campos nuevos (vacío -> None)
def fecha_comparable(x):
    # Fecha en formato AAAA-MM-DD; vacío -> None.
    if pd.isna(x):
        return None
    return parse_birth_date(x) or (clean_text(x) or None)


def texto_comparable(x):
    # Texto en minúsculas sin espacios extra; vacío -> None.
    if pd.isna(x):
        return None
    return clean_text(x).casefold() or None

**Comentario 3:** Estas funciones son la base de todas las comparaciones posteriores: `clean_text` quita comillas, espacios sobrantes y saltos de línea, `parse_birth_date` reconoce las tres formas de escribir una fecha que existen en los datos, `texto_comparable` y `fecha_comparable` llevan los textos a minúsculas y las fechas a `AAAA-MM-DD`. Sin esta normalización, dos valores idénticos escritos distinto se contarían como diferentes y se detectarían **falsas inconsistencias**.

In [19]:
# AUDITORÍA DE FORMATO DE TEXTO EN LA FICHA NOMINAL
# Se busca texto con comillas, saltos de línea, espacios extra,
# diferencias de mayúsculas/minúsculas o de tildes que hacen que un mismo valor aparezca escrito de varias formas.

def forma_canonica(valor):
    # Forma comparable de un texto: sin comillas, espacios uniformes, sin tildes y en minúsculas.
    texto = unicodedata.normalize("NFKD", clean_text(valor))
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    return texto.casefold()

resumen_formato, detalle_variantes = [], []
for posicion, columna in enumerate(df_nominal.columns):
    serie = df_nominal[columna]
    if not serie.map(lambda v: isinstance(v, str)).all():
        continue                                   # solo columnas de texto
    visible = serie.map(clean_text)                # texto visible: sin comillas ni espacios sobrantes
    canonico = serie.map(forma_canonica)
    resumen_formato.append({
        "posicion": posicion, "columna": columna,
        "valores_distintos_originales": serie.nunique(),
        "valores_distintos_visibles": visible.nunique(),
        "valores_distintos_tras_unificar": canonico.nunique(),
        "celdas_con_comillas": int(serie.str.startswith('"').sum()),
        "celdas_con_saltos_de_linea": int(serie.str.contains(r"[\r\n]").sum()),
    })
    # Valores que solo difieren en mayúsculas, tildes o espacios (mismo canónico, distinto texto visible)
    for forma, grupo in visible.groupby(canonico):
        if grupo.nunique() > 1:
            detalle_variantes.append({
                "columna": columna, "forma_comun": forma,
                "variantes": " | ".join(f"{valor!r} ({cuenta})" for valor, cuenta in grupo.value_counts().items()),
                "filas_afectadas": len(grupo)})

resumen_formato = pd.DataFrame(resumen_formato).set_index("posicion")
detalle_variantes = pd.DataFrame(detalle_variantes)

show_table(resumen_formato[["columna", "valores_distintos_originales", "valores_distintos_visibles",
                            "valores_distintos_tras_unificar", "celdas_con_comillas", "celdas_con_saltos_de_linea"]],
           "Formato de texto por columna de la ficha nominal")

,columna,valores_distintos_originales,valores_distintos_visibles,valores_distintos_tras_unificar,celdas_con_comillas,celdas_con_saltos_de_linea
posicion,,,,,,
0,periodo,19,19,19,2667,0
1,IDs_FichaNominal,157,157,157,0,0
2,Sexo,2,2,2,2667,0
3,Fecha de nacimiento,154,154,154,2667,0
4,Departamento,8,8,8,2667,0
5,Provincia,12,12,12,2667,0
6,Distrito,33,33,33,2667,0
7,Nacionalidad,2,2,2,2667,0
8,Fecha de ingreso,107,107,107,2667,0


Los encabezados son:
* posicion: Número de la columna en la ficha nominal, contando desde 0.
* columna: Nombre de la columna.
* valores_distintos_originales: Valores distintos tal como llegaron en el Excel, con comillas, espacios, mayúsculas y saltos de línea.
* valores_distintos_visibles: Valores distintos después de clean_text, que quita las comillas de los bordes, los saltos de línea y los espacios sobrantes.
* valores_distintos_tras_unificar: Valores distintos después de quitar además las tildes y pasar todo a minúsculas (forma_canonica).
* celdas_con_comillas: Cuántas celdas empiezan con ".
* celdas_con_saltos_de_linea: Cuántas celdas tienen un salto de línea dentro del texto.

In [22]:
show_table(detalle_variantes.set_index("columna"), "Mismo valor escrito de varias formas (mayúsculas, tildes o espacios)")

# Exportar la evidencia de lo encontrado
export_evidence("01_formato_texto_ficha_nominal.xlsx", {
    "resumen_por_columna": resumen_formato.reset_index(),
    "variantes_de_un_mismo_valor": detalle_variantes})

,forma_comun,variantes,filas_afectadas
columna,,,
Observaciones y/o comentarios de la forma de ingreso de la NNA,proteccion provisional,'Proteccion Provisional' (57) | 'Proteccion provisional' (19),76
VIGENCIA DEL PTI,mientras dure su proceso legal para su adoptabilidad,'mientras dure su proceso legal para su adoptabilidad' (1) | 'Mientras dure su proceso legal para su adoptabilidad' (1),2
VIGENCIA DEL PTI,no tiene,'No tiene' (1019) | 'no tiene' (14),1033
ESTADO DEL PTI,no tiene,'No tiene' (999) | 'no tiene' (33),1032
Grado de Instrucción,1er grado de primaria,'1er Grado de primaria' (213) | '1er Grado de Primaria' (3),216
Grado de Instrucción,2do de secundaria,'2do de Secundaria' (212) | '2do de secundaria' (5),217
Grado de Instrucción,2do grado de primaria,'2do Grado de Primaria' (77) | '2do Grado de primaria' (8) | '2Do Grado de Primaria' (1),86
Grado de Instrucción,3er grado de primaria,'3er Grado de primaria' (99) | '3er Grado de Primaria' (26),125
Grado de Instrucción,3ero de secundaria,'3ero de Secundaria' (42) | '3ero de secundaria' (2),44


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/01_formato_texto_ficha_nominal.xlsx  (resumen_por_columna: 32 filas, variantes_de_un_mismo_valor: 29 filas)


Columnas trabajadas:
* columna: Columna de la ficha nominal donde ocurre el problema. Es el índice, por eso va a la izquierda en negrita.
* forma_comun: El valor «limpio» que comparten todas las variantes: sin comillas, sin tildes y en minúsculas. Sirve solo para agruparlas; no es un valor real del archivo.
* variantes: Cómo está escrito realmente ese valor en el Excel, con cuántas filas usa cada escritura entre paréntesis.
* filas_afectadas: Suma de filas de todas las variantes de ese grupo.


Si se agrupa o cuenta por estas columnas, un mismo estado aparecería partido en varias categorías (por ejemplo, el estado del PTI).

Estamos evaluando las cantidades que quedarian de datos por columna luego de aplicar los filtros correspondientes para poder identificar con cuántos datos nos estamos quedando si se hiciera ese tratamiento y evaluar si es viable.

In [23]:
# FORMATO DE LAS FECHAS EN LA FICHA NOMINAL
# Se reemplazan los dígitos por 9 para ver los "moldes" de escritura que conviven en una misma columna.
columnas_fecha = [3, 8, 12, 14, 17]     # nacimiento, ingreso, resolución, oficio, elaboración del PTI
patrones_fecha = []
for pos in columnas_fecha:
    columna = df_nominal.columns[pos]
    moldes = df_nominal[columna].map(lambda v: re.sub(r"\d", "9", clean_text(v)))
    for molde, cuenta in moldes.value_counts().items():
        patrones_fecha.append({"columna": columna, "molde_de_escritura": molde, "filas": cuenta})
patrones_fecha = pd.DataFrame(patrones_fecha)

show_table(patrones_fecha.set_index("columna"), "Moldes de escritura de las fechas")

# Exportar la evidencia de los casos encontrados
export_evidence("02_formato_fechas_ficha_nominal.xlsx", {"moldes_de_fecha": patrones_fecha})

,molde_de_escritura,filas
columna,,
Fecha de nacimiento,9999-99-99,2560
Fecha de nacimiento,99/99/9999,107
Fecha de ingreso,9999-99-99,1739
Fecha de ingreso,99/99/9999,928
Fecha de la Resolucion Administrativa,9999-99-99,2002
Fecha de la Resolucion Administrativa,99/99/9999,464
Fecha de la Resolucion Administrativa,NA,182
Fecha de la Resolucion Administrativa,99/99/99,19
Fecha del Oficio de Ingreso,9999-99-99,2183


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/02_formato_fechas_ficha_nominal.xlsx  (moldes_de_fecha: 13 filas)


**Comentario 5:** Las columnas de fecha mezclan formatos en una misma columna:

- `Fecha de nacimiento`: 2 560 filas en `AAAA-MM-DD` y 107 en `DD/MM/AAAA`.
- `Fecha de ingreso`: 1 739 en `AAAA-MM-DD` y 928 en `DD/MM/AAAA`.
- `Fecha de la Resolución Administrativa`: 2 002 en `AAAA-MM-DD`, 464 en `DD/MM/AAAA`, 19 en `DD/MM/AA` (año de dos cifras) y 182 con el texto `NA`.
- `Fecha del Oficio de Ingreso`: 2 183 / 426 / 58 `NA`.
- `FECHA ELABORACIÓN … PTI`: 1 635 fechas y 1 032 con el texto `No tiene`.

**Riesgo:** una fecha como `09/01/2023` puede leerse como 9 de enero o 1 de septiembre.

# 3. Códigos (IDs) con datos contradictorios en la ficha nominal

Para que una evaluación de nutrición pueda unirse a una ficha nominal, el código del NNA debe identificar siempre a la misma persona. Un código cuya fecha de nacimiento, sexo, nacionalidad o datos de ingreso cambian entre meses no es confiable para vincular registros.

In [24]:
# SELECCIÓN DE CAMPOS PARA VALIDAR LA IDENTIDAD DE LOS NNA
# Posiciones (desde 0): 0 periodo, 1 id, 2 sexo, 3 fecha de nacimiento,
# 7 nacionalidad, 8 fecha de ingreso, 9 observaciones forma de ingreso, 14 fecha del oficio de ingreso

columnas = [0, 1, 2, 3, 7, 8, 9, 14]
revision = df_nominal.iloc[:, columnas].copy()
print("Columnas seleccionadas:", list(revision.columns))   # Verifica que sean las correctas
revision.columns = ["periodo", "id_persona", "sexo", "fecha_nacimiento",
                    "nacionalidad", "fecha_ingreso", "obs_forma_ingreso", "fecha_oficio_ingreso"]


# Normalizar los valores utilizando las funciones originales
revision["periodo"] = revision["periodo"].apply(
    lambda x: clean_text(x).replace("_", "-")
)
revision["id_persona"] = revision["id_persona"].apply(clean_text)
revision["sexo"] = revision["sexo"].apply(clean_text)
revision["fecha_nacimiento"] = revision["fecha_nacimiento"].apply(parse_birth_date)
revision["nacionalidad"] = revision["nacionalidad"].apply(texto_comparable)
revision["fecha_ingreso"] = revision["fecha_ingreso"].apply(fecha_comparable)
revision["obs_forma_ingreso"] = revision["obs_forma_ingreso"].apply(texto_comparable)
revision["fecha_oficio_ingreso"] = revision["fecha_oficio_ingreso"].apply(fecha_comparable)


# AGRUPAR POR NNA Y EVALUAR LA CONSISTENCIA
resultado = revision.groupby("id_persona").agg(
    registros=("periodo", "size"),
    nacimientos_distintos=("fecha_nacimiento", lambda x: x.nunique(dropna=False)),
    sexos_distintos=("sexo", "nunique"),
    nacionalidad_distintas=("nacionalidad", "nunique"),
    fecha_ingreso_distintas=("fecha_ingreso", "nunique"),
    fecha_oficio_distintas=("fecha_oficio_ingreso", "nunique"),
    obs_forma_ingreso_distintas=("obs_forma_ingreso", "nunique")
)

# CONTAR LOS MESES CON MÁS DE UNA FICHA
duplicados = (
    revision.groupby(["id_persona", "periodo"])
    .size()
    .gt(1)
    .groupby(level=0)
    .sum()
)
resultado["meses_con_varias_fichas"] = duplicados


# IDENTIFICAR LOS CÓDIGOS CON INCONSISTENCIAS
inconsistentes = resultado[
    (resultado["nacimientos_distintos"] > 1) |
    (resultado["sexos_distintos"] > 1) |
    (resultado["meses_con_varias_fichas"] > 0) |
    (resultado["nacionalidad_distintas"] > 1) |
    (resultado["fecha_ingreso_distintas"] > 1) |
    (resultado["fecha_oficio_distintas"] > 1) |
    (resultado["obs_forma_ingreso_distintas"] > 1)
]


# Mostrar los resultados
print("Identificadores que requieren revisión:", len(inconsistentes))
display(inconsistentes)

Columnas seleccionadas: ['periodo', 'IDs_FichaNominal', 'Sexo', 'Fecha de nacimiento', 'Nacionalidad', 'Fecha de ingreso', 'Observaciones y/o comentarios de la forma de ingreso de la NNA', 'Fecha del Oficio de Ingreso']
Identificadores que requieren revisión: 8


,registros,nacimientos_distintos,sexos_distintos,nacionalidad_distintas,fecha_ingreso_distintas,fecha_oficio_distintas,obs_forma_ingreso_distintas,meses_con_varias_fichas
id_persona,,,,,,,,
PPA_FN_0010,19,2,1,1,1,1,1,0
PPA_FN_0020,36,5,2,2,5,5,1,9
PPA_FN_0025,19,2,1,1,1,1,1,0
PPA_FN_0035,19,1,1,1,2,1,1,0
PPA_FN_0036,19,1,1,1,2,1,1,0
PPA_FN_0071,19,1,1,1,2,1,1,0
PPA_FN_0082,19,1,1,1,2,2,1,0
PPA_FN_0127,19,2,1,1,1,1,1,0


**Comentario 6:** De los 157 códigos de la ficha nominal, 8 presentan inconsistencias. El criterio es: más de un valor distinto de fecha de nacimiento, sexo, nacionalidad, fecha de ingreso, fecha de oficio u observación de ingreso, o más de una ficha en el mismo mes, o más de 19 registros (el máximo esperado en 19 meses).

In [26]:
# Obtener los identificadores inconsistentes detectados previamente
conflicting_ids = set(inconsistentes.index)

print("Identificadores que requieren revisión:")
print(sorted(conflicting_ids))

# DETALLE: ¿QUÉ CAMPOS DIFIEREN EN CADA CÓDIGO INCONSISTENTE?
motivos = []
for ident, fila in inconsistentes.iterrows():
    razones = []
    if fila["nacimientos_distintos"] > 1: razones.append(f'{fila["nacimientos_distintos"]} fechas de nacimiento')
    if fila["sexos_distintos"] > 1: razones.append(f'{fila["sexos_distintos"]} valores de sexo')
    if fila["nacionalidad_distintas"] > 1: razones.append(f'{fila["nacionalidad_distintas"]} nacionalidades')
    if fila["fecha_ingreso_distintas"] > 1: razones.append(f'{fila["fecha_ingreso_distintas"]} fechas de ingreso')
    if fila["fecha_oficio_distintas"] > 1: razones.append(f'{fila["fecha_oficio_distintas"]} fechas de oficio')
    if fila["obs_forma_ingreso_distintas"] > 1: razones.append(f'{fila["obs_forma_ingreso_distintas"]} observaciones de ingreso')
    if fila["meses_con_varias_fichas"] > 0: razones.append(f'{fila["meses_con_varias_fichas"]} meses con varias fichas')
    if fila["registros"] > 19: razones.append(f'{fila["registros"]} registros (máximo esperado: 19)')
    motivos.append("; ".join(razones))

ids_inconsistentes_detalle = inconsistentes.copy()
ids_inconsistentes_detalle["motivos"] = motivos
show_table(ids_inconsistentes_detalle[["registros", "motivos"]], "Motivo de inconsistencia por código")

# Filas de la ficha nominal que pertenecen a esos códigos (con su número de fila en el Excel original)
filas_ids_inconsistentes = revision[revision["id_persona"].isin(conflicting_ids)].copy()
filas_ids_inconsistentes.insert(0, "fila_excel_origen", filas_ids_inconsistentes.index + 2)
filas_ids_inconsistentes = filas_ids_inconsistentes.sort_values(["id_persona", "periodo"])

# Exportar la evidencia encontrada
export_evidence("03_ids_inconsistentes.xlsx", {
    "resumen_por_id": ids_inconsistentes_detalle.reset_index(),
    "filas_nominales_afectadas": filas_ids_inconsistentes})

Identificadores que requieren revisión:
['PPA_FN_0010', 'PPA_FN_0020', 'PPA_FN_0025', 'PPA_FN_0035', 'PPA_FN_0036', 'PPA_FN_0071', 'PPA_FN_0082', 'PPA_FN_0127']


,registros,motivos
id_persona,,
PPA_FN_0010,19,2 fechas de nacimiento
PPA_FN_0020,36,5 fechas de nacimiento; 2 valores de sexo; 2 nacionalidades; 5 fechas de ingreso; 5 fechas de oficio; 9 meses con varias fichas; 36 registros (máximo esperado: 19)
PPA_FN_0025,19,2 fechas de nacimiento
PPA_FN_0035,19,2 fechas de ingreso
PPA_FN_0036,19,2 fechas de ingreso
PPA_FN_0071,19,2 fechas de ingreso
PPA_FN_0082,19,2 fechas de ingreso; 2 fechas de oficio
PPA_FN_0127,19,2 fechas de nacimiento


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/03_ids_inconsistentes.xlsx  (resumen_por_id: 8 filas, filas_nominales_afectadas: 169 filas)


**Comentario 7:**

- **PPA_FN_0020 es el caso crítico:** 36 registros (máximo esperado 19), 5 fechas de nacimiento, 2 valores de sexo, 2 nacionalidades, 5 fechas de ingreso, 5 fechas de oficio y 9 meses con más de una ficha. Esto sugiere que más de una persona distinta podría estar registrada bajo este código.
- **PPA_FN_0010, 0025 y 0127:** dos fechas de nacimiento distintas.
- **PPA_FN_0035, 0036 y 0071:** dos fechas de ingreso; **PPA_FN_0082:** dos fechas de ingreso y dos fechas de oficio.

En total afectan 169 fichas nominales. Mientras no se sepa el motivo, vincular nutrición con estos códigos podría asignar una evaluación a la persona equivocada. Por eso es conveniente excluirlos del consolidado. El Excel `03_ids_inconsistentes.xlsx` guarda el resumen por código y las filas afectadas con su número de fila en el archivo original, para que la institución pueda corregirlas en la fuente.

# 4. Plantillas de los Excel de nutrición

Los libros de nutrición no son una tabla única, ya que cada mes y cada hogar usa su propia hoja, con encabezados en varias filas, celdas combinadas y una sección por casa. Antes de extraer datos hay que saber qué columnas tiene cada hoja.

In [28]:
def inspect_nutrition_layout(source):
    rows = source["rows"]
    headers = rows[2]
    groups = {}
    for merged_range in source["merged_ranges"]:
        min_col, min_row, max_col, max_row = openpyxl.utils.range_boundaries(merged_range)
        if min_row == 2:
            title = rows[1][min_col - 1]
            for column in range(min_col, max_col + 1):
                groups[column] = title
    person_rows = [row for row in rows if len(row) > 1 and
                   re.fullmatch(r"PPA_FN_\d+", str(row[1]))]
    columns = []
    for index, header in enumerate(headers, start=1):
        if header is not None:
            values = [row[index - 1] if index <= len(row) else None for row in person_rows]
            columns.append({
                "column": get_column_letter(index),
                "header": re.sub(r"\s+", " ", str(header)).strip(),
                "group": groups.get(index),
                "types": Counter(type(value).__name__ for value in values)
            })
    return {"sheet": source["sheet"], "records": len(person_rows), "columns": columns}

nutrition_layout = [
    {"file": filename, "sheets": [inspect_nutrition_layout(sheet) for sheet in sheets]}
    for filename, sheets in nutrition_sources.items()
]
layout_review = []
for book in nutrition_layout:
    period_digits = re.search(r"2026\d{2}", book["file"]).group()
    period = period_digits[:4] + "-" + period_digits[4:]
    for sheet in book["sheets"]:
        layout_review.append({
            "periodo": period, "hoja": sheet["sheet"], "registros": sheet["records"],
            "encabezados": "; ".join(f'{c["column"]}: {c["header"]}' for c in sheet["columns"])
        })
show_table(pd.DataFrame(layout_review), "Plantillas nutricionales observadas")

# Exportar evidencia
# export_evidence("04_plantillas_nutricion.xlsx", {"plantillas_observadas": pd.DataFrame(layout_review)})

,periodo,hoja,registros,encabezados
0,2026-01,INFANTES,33,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: P/E; I: T/E; J: P/T; K: IMC; L: P/E; M: T/E; N: P/T; O: IMC
1,2026-01,MIGUEL ECHENIQUE,17,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: T/E; I: IMC/E; J: T/E; K: IMC/E
2,2026-01,TOMAS VALLE,22,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: T/E; I: IMC/E; J: T/E; K: IMC/E
3,2026-01,MATOGROSO MUJERES,32,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: T/E; I: IMC/E; J: T/E; K: IMC
4,2026-01,MATOGROSO HOMBRES,37,A: N; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: T/E; I: IMC/E; J: T/E; K: IMC
5,2026-02,Eva.Nut febrero,7,A: N; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: P/E; I: T/E; J: P/T; K: IMC; L: P/E; M: T/E; N: P/T; O: IMC
6,2026-03,Eva.Nut marzo,6,A: N; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: P.CEFALICO; H: P/E; I: T/E; J: P/T; K: IMC; L: P/E; M: T/E; N: P/T; O: IMC
7,2026-04,INFANTES (ABR),30,A: N; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: P/E; I: T/E; J: P/T; K: IMC; L: P/E; M: T/E; N: P/T; O: IMC
8,2026-04,MIGUEL ECHENIQUE,16,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: PA; H: IMC; I: T/E; J: IMC/E; K: T/E; L: IMC/E
9,2026-04,TOMAS VALLE,20,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: P.A; H: IMC; I: P/E; J: T/E; K: IMC/E; L: T/E; M: IMC/E


**Comentario 8:** Hallazgos de estructura:

- **Hogares de infantes** (INFANTES, febrero, marzo): usan los tres indicadores del menor de 5 años, **P/E, T/E y P/T**, más IMC, en **dos bloques** («Según percentiles OMS» y «Según tabla de diagnóstico rápido INS»).
- **Hogares de escolares y adolescentes** (Miguel Echenique, Tomás Valle, Matogroso): solo **T/E e IMC/E** en cada bloque. En abril y junio algunas hojas agregan también **P/E**, que no corresponde a esa edad.
- **Marzo** trae **P. CEFÁLICO** en lugar de IMC, por lo que el IMC no fue reportado en esa hoja.
- **Abril** incluye una columna **PA / P.A** de significado y unidad no declarados.
- **Agosto** agrega una columna adicional de edad (`EDAD C/ FORMULA`) además de `EDAD`.
- El título del segundo bloque no es uniforme («Según percentiles OMS», «Según puntaje z», «Según tabla de diagnóstico rápido…»). En particular, **junio – Miguel Echenique rotula su primer bloque como «puntaje z»**, mientras el resto dice «percentiles»

**Consecuencia:** hay que identificar cada columna por su encabezado y por el título combinado que la agrupa, no por su posición.

# 5. Extracción de las evaluaciones de nutrición (para analizarlas)

Se extraen las filas con códigos `PPA_FN_…` y se asocian con su mes, hogar, casa y coordenadas de origen. Se conservan el valor tratado y el valor original de cada celda (columnas `*_original`) y su ubicación exacta (archivo, hoja, fila), lo que permite auditar cualquier dato.

In [29]:
text_corrections = {
    "NORNAL": "NORMAL", "OBEESIDAD": "OBESIDAD", "OBESIDA": "OBESIDAD",
    "RIESO TALLA BAJA": "RIESGO TALLA BAJA", "RIESGO TALA BAJA": "RIESGO TALLA BAJA",
    "RIESGO DE TALLA BAJA": "RIESGO TALLA BAJA",
    "RIESGO DE SOBREPESO": "RIESGO SOBREPESO",
    "RIESGO SOBRE PESO": "RIESGO SOBREPESO", "SOBRE PESO": "SOBREPESO",
    "RIEGO DE DELGADEZ O RIESGO DE BAJO PESO": "RIESGO DE DELGADEZ O RIESGO DE BAJO PESO"
}

def normalize_diagnosis(value):
    text = clean_text(value).upper()
    return text_corrections.get(text, text) if text not in ("", "*") else None

show_table(pd.DataFrame(text_corrections.items(), columns=["texto", "normalización"]).set_index("texto"),
           "Correcciones textuales explícitas")

,normalización
texto,
NORNAL,NORMAL
OBEESIDAD,OBESIDAD
OBESIDA,OBESIDAD
RIESO TALLA BAJA,RIESGO TALLA BAJA
RIESGO TALA BAJA,RIESGO TALLA BAJA
RIESGO DE TALLA BAJA,RIESGO TALLA BAJA
RIESGO DE SOBREPESO,RIESGO SOBREPESO
RIESGO SOBRE PESO,RIESGO SOBREPESO
SOBRE PESO,SOBREPESO


In [30]:
records = []
issues = []
notes = []
normalization_counts = Counter()
def issue(rec, field, kind, detail, cell=None):
    issues.append({'id_evaluacion':rec['id_evaluacion'],'campo':field,'observacion':kind,'detalle':detail,
        'archivo_origen':rec['archivo_origen'],'hoja_origen':rec['hoja_origen'],'celda_origen':cell or str(rec['fila_origen'])})

In [31]:
# DICCIONARIO

from collections import defaultdict

# Preparar las claves nominales desde el DataFrame
nominal_by_key = defaultdict(list)
nominal_by_id = defaultdict(list)

for fila, row in df_nominal.iterrows():

    periodo = clean_text(row.iloc[0]).replace("_", "-")
    ident = clean_text(row.iloc[1])

    registro = {
        "fila": fila + 2,
        "periodo": periodo,
        "id": ident,
        "sexo": clean_text(row.iloc[2]),
        "dob": parse_birth_date(row.iloc[3])
    }

    nominal_by_key[(periodo, ident)].append(registro)
    nominal_by_id[ident].append(registro)

# Identificar los meses disponibles en la ficha nominal
nominal_periods = {
    clean_text(valor).replace("_", "-")
    for valor in df_nominal.iloc[:, 0]
}

print("Claves nominales preparadas desde df.")
print("Identificadores:", len(nominal_by_id))
print("Periodos:", len(nominal_periods))

Claves nominales preparadas desde df.
Identificadores: 157
Periodos: 19


In [32]:
# EXTRACCIÓN DE EVALUACIONES NUTRICIONALES

for f in nutrition_layout:

    path = nutrition_dir / f["file"]

    # Obtener el periodo desde el nombre del archivo
    period = re.search(r"2026\d{2}", path.name).group()
    period = period[:4] + "-" + period[4:]

    for s in f["sheets"]:

        # Obtener los datos originales de la hoja
        source = next(
            sheet for sheet in nutrition_sources[path.name]
            if sheet["sheet"] == s["sheet"]
        )

        rows = source["rows"]
        formulas = source["formulas"]
        hdr = rows[2]

        # Identificar las columnas de peso y talla
        wc = next(i for i, h in enumerate(hdr)
                  if h and "PESO" in str(h))
        hc = next(i for i, h in enumerate(hdr)
                  if h and "TALLA" in str(h))

        # Identificar las columnas de diagnósticos nutricionales
        diag = [
            c for c in s["columns"]
            if c["header"] in ("P/E", "T/E", "P/T", "IMC/E", "IMC")
            and c["group"] != "DATOS"
            and c["types"].get("str", 0) > 0
        ]

        # Clasificar diagnósticos según curva o tabla
        blockmap = {}
        seen = Counter()

        for c in diag:
            indicator = {
                "P/E": "pe",
                "T/E": "te",
                "P/T": "pt",
                "IMC/E": "imc",
                "IMC": "imc"
            }[c["header"]]

            seen[indicator] += 1
            block = "curva" if seen[indicator] == 1 else "tabla"

            if c["group"] and (
                "TABLA" in clean_text(c["group"]).upper()
                or "RAPIDO" in clean_text(c["group"]).upper()
            ):
                block = "tabla"

            blockmap[(indicator, block)] = c

        # Identificar columnas opcionales según la plantilla
        bmi = next(
            (column_index_from_string(c["column"]) - 1
             for c in s["columns"]
             if c["header"] == "IMC" and c not in diag),
            None
        )

        pc = next(
            (column_index_from_string(c["column"]) - 1
             for c in s["columns"]
             if c["header"] == "P.CEFALICO"),
            None
        )

        pa = next(
            (column_index_from_string(c["column"]) - 1
             for c in s["columns"]
             if c["header"] in ("PA", "P.A")),
            None
        )

        # Identificar el grupo y la casa
        group = (
            "INFANTES"
            if "INFANTES" in s["sheet"].upper()
            or s["sheet"].startswith("Eva.Nut")
            else s["sheet"].strip()
        )

        house = None

        # Recorrer los registros de la hoja nutricional
        for rn, r in enumerate(rows, 1):

            get = lambda ix: (
                r[ix] if ix is not None and ix < len(r) else None
            )

            fg = lambda ix: (
                formulas[rn - 1][ix]
                if ix is not None and ix < len(formulas[rn - 1])
                else None
            )

            # Identificar la casa a la que pertenecen los registros
            if (
                rn > 3
                and isinstance(get(0), str)
                and clean_text(get(0)).upper().startswith("CASA")
            ):
                house = clean_text(get(0))
                continue

            # Conservar las filas sin identificador como notas
            if not re.fullmatch(r"PPA_FN_\d+", str(get(1))):
                if rn > 3 and any(v is not None for v in r):
                    notes.append({
                        "periodo": period,
                        "grupo_hogar": group,
                        "archivo_origen": path.name,
                        "hoja_origen": s["sheet"],
                        "fila_origen": rn,
                        "nota_original": " / ".join(
                            f"{get_column_letter(i + 1)}{rn}: {serialize_original(v)}"
                            for i, v in enumerate(r)
                            if v is not None
                        )
                    })
                continue

            # Identificación de la evaluación
            ident = get(1)
            eid = f'NUT_{period.replace("-", "")}_{ident}'

            # Obtener mediciones originales
            w = get(wc)
            h = get(hc)
            born = get(2)

            # Identificar la estructura especial de edad en agosto
            aug = clean_text(hdr[3]) == "EDAD C/ FORMULA"
            ageix = 4 if aug else 3
            ageformula = fg(3)

            isageformula = (
                isinstance(ageformula, str)
                and ageformula.startswith("=")
            )

            # Crear el registro nutricional
            rec = {
                "id_evaluacion": eid,
                "id_persona": ident,
                "periodo": period,
                "grupo_hogar": group,
                "casa": house,

                # Medidas nutricionales
                "peso_kg": numeric_value(w),
                "talla_cm": numeric_value(h),
                "imc_reportado": numeric_value(get(bmi)),
                "imc_calculado": None,
                "diferencia_imc": None,

                "estado_mediciones": (
                    "sin_medicion_provincia"
                    if clean_text(w) == "PROVINCIA"
                    else "peso_talla_registrados"
                    if numeric_value(w) is not None
                    and numeric_value(h) is not None
                    else "mediciones_incompletas"
                ),

                # Información de nacimiento y edad
                "fecha_nacimiento": parse_birth_date(born),
                "fecha_evaluacion": None,
                "edad_reportada": (
                    serialize_original(get(ageix))
                    if aug or not isageformula else None
                ),
                "edad_formula_guardada": (
                    serialize_original(get(3))
                    if isageformula or aug else None
                ),

                # Información nominal pendiente de vinculación
                "sexo_nominal": None,
                "estado_vinculo_nominal": None,

                # Métodos de evaluación nutricional
                "metodo_curva_original": next(
                    (c["group"] for (ind, bl), c in blockmap.items()
                     if bl == "curva" and c["group"]),
                    None
                ),

                "metodo_tabla_original": next(
                    (c["group"] for (ind, bl), c in blockmap.items()
                     if bl == "tabla" and c["group"]),
                    None
                ),

                "indicador_imc_curva_original": (
                    blockmap.get(("imc", "curva"), {}).get("header")
                ),
                "indicador_imc_tabla_original": (
                    blockmap.get(("imc", "tabla"), {}).get("header")
                ),

                # Valores originales para trazabilidad
                "peso_original": serialize_original(w),
                "talla_original": serialize_original(h),
                "imc_original": serialize_original(get(bmi)),
                "imc_formula_original": (
                    fg(bmi) if isinstance(fg(bmi), str)
                    and fg(bmi).startswith("=") else None
                ),

                "perimetro_cefalico_original": serialize_original(get(pc)),
                "perimetro_cefalico_unidad": (
                    "no_declarada" if pc is not None else None
                ),

                "pa_original": serialize_original(get(pa)),
                "pa_encabezado_original": (
                    hdr[pa] if pa is not None else None
                ),
                "pa_unidad": "no_declarada" if pa is not None else None,

                "fecha_nacimiento_original": serialize_original(born),
                "edad_formula_original": (
                    ageformula if isageformula else None
                ),
                "numero_original": serialize_original(get(0)),
                "observacion_original": (
                    str(w) if numeric_value(w) is None
                    and clean_text(w) else None
                ),

                # Procedencia del registro
                "archivo_origen": path.name,
                "hoja_origen": s["sheet"],
                "fila_origen": rn,

                "columnas_diagnostico_originales": "; ".join(
                    f'{i}_{b}:{c["column"]}'
                    for (i, b), c in blockmap.items()
                ),

                # Vinculación con la ficha nominal
                "sexo_archivo_origen": None,
                "sexo_hoja_origen": None,
                "sexo_fila_origen": None,
                "fecha_nacimiento_nominal": None,

                "coincidencias_nominal_mes": len(
                    nominal_by_key.get((period, ident), [])
                ),

                "identidad_nominal_observacion": None,

                # Indicadores de calidad
                "asteriscos_diagnostico": 0,
                "vacios_diagnostico": 0,
                "indicadores_ausentes": 0,
                "normalizaciones_texto": 0,
                "diferencias_entre_metodos": 0,
                "observaciones_calidad": None
            }

            # PROCESAMIENTO DE DIAGNÓSTICOS NUTRICIONALES

            for block in ("curva", "tabla"):
                for indicator in ("pe", "te", "pt", "imc"):

                    key = f"dx_{indicator}_{block}"
                    c = blockmap.get((indicator, block))

                    v = (
                        get(column_index_from_string(c["column"]) - 1)
                        if c else None
                    )

                    # Identificar el estado original del dato
                    state = (
                        "ausente_en_plantilla" if not c
                        else "vacio_original" if clean_text(v) == ""
                        else "asterisco_original" if clean_text(v) == "*"
                        else "categoria_registrada"
                    )

                    rec[key] = normalize_diagnosis(v)
                    rec[key + "_original"] = serialize_original(v)
                    rec[key + "_estado"] = state

                    if state == "asterisco_original":
                        rec["asteriscos_diagnostico"] += 1
                    elif state == "vacio_original":
                        rec["vacios_diagnostico"] += 1
                    elif state == "ausente_en_plantilla":
                        rec["indicadores_ausentes"] += 1

                    # Registrar normalizaciones realizadas
                    if v is not None and state == "categoria_registrada":
                        normalization_counts[
                            (str(v), normalize_diagnosis(v))
                        ] += 1

                        if str(v) != normalize_diagnosis(v):
                            rec["normalizaciones_texto"] += 1

            # Comparar diagnósticos entre los dos métodos
            for indicator in ("pe", "te", "pt", "imc"):
                v1 = rec[f"dx_{indicator}_curva"]
                v2 = rec[f"dx_{indicator}_tabla"]

                if v1 is not None and v2 is not None and v1 != v2:
                    rec["diferencias_entre_metodos"] += 1

            # VINCULACIÓN CON LA FICHA NOMINAL

            # Buscar el identificador y periodo en la ficha nominal
            nr = nominal_by_key.get((period, ident), [])

            # NUEVO: aprovechar los identificadores inconsistentes
            # detectados previamente mediante pandas
            if ident in conflicting_ids:
                rec["identidad_nominal_observacion"] = (
                    "identificador_con_datos_contradictorios"
                )

            # Determinar si se puede realizar una unión segura
            if not nr:
                rec["estado_vinculo_nominal"] = (
                    "periodo_nominal_no_disponible"
                    if period not in nominal_periods
                    else "id_ausente_en_mes_nominal"
                )

            elif len(nr) > 1:
                rec["estado_vinculo_nominal"] = "multiples_coincidencias"

            elif nr[0]["dob"] != rec["fecha_nacimiento"]:
                rec["estado_vinculo_nominal"] = (
                    "fecha_nacimiento_no_coincide"
                )
                rec["fecha_nacimiento_nominal"] = nr[0]["dob"]

            elif rec["identidad_nominal_observacion"]:
                rec["estado_vinculo_nominal"] = (
                    "identidad_contradictoria_no_vinculada"
                )
                rec["fecha_nacimiento_nominal"] = nr[0]["dob"]

            else:
                # Unión segura: se obtiene el sexo de la ficha nominal
                rec.update(
                    sexo_nominal=nr[0]["sexo"],
                    estado_vinculo_nominal=(
                        "unico_mes_y_nacimiento_coinciden"
                    ),
                    sexo_archivo_origen=nominal_path.name,
                    sexo_hoja_origen=nominal_sheet_name,
                    sexo_fila_origen=nr[0]["fila"],
                    fecha_nacimiento_nominal=nr[0]["dob"]
                )

            # CONTROL DE CALIDAD DE LA EVALUACIÓN

            flags = []

            if rec["estado_mediciones"] != "peso_talla_registrados":
                flags.append("sin peso/talla: PROVINCIA")

            if rec["asteriscos_diagnostico"]:
                flags.append("asteriscos sin leyenda")

            if rec["vacios_diagnostico"]:
                flags.append("diagnosticos vacios")

            if rec["metodo_curva_original"] is None:
                flags.append("metodo curva sin titulo")

                issue(
                    rec,
                    "metodo_curva_original",
                    "titulo_no_declarado",
                    "El bloque 1 existe pero no declara metodo en su encabezado.",
                    "I2:J2"
                )

            if pa is not None:
                flags.append("PA: significado/unidad por confirmar")

            if isageformula and "TODAY" in ageformula.upper():
                flags.append("edad calculada con TODAY en origen")

            if rec["estado_vinculo_nominal"] != (
                "unico_mes_y_nacimiento_coinciden"
            ):
                flags.append("sexo nominal no vinculado")

                issue(
                    rec,
                    "sexo_nominal",
                    rec["estado_vinculo_nominal"],
                    "No se completa sexo mediante una union ambigua, "
                    "otro mes o fecha de nacimiento distinta."
                )

            # Verificar diferencias entre el IMC calculado y reportado
            calc = (
                numeric_value(w) / (numeric_value(h) / 100) ** 2
                if numeric_value(w) is not None
                and numeric_value(h) is not None
                and numeric_value(h) > 0
                else None
            )

            if (
                calc is not None
                and rec["imc_reportado"] is not None
                and abs(rec["imc_reportado"] - calc) > 0.11
            ):
                flags.append("IMC reportado difiere >0.11")

                issue(
                    rec,
                    "imc_reportado",
                    "diferencia_imc",
                    "Reportado menos peso/(talla/100)^2: "
                    + str(round(rec["imc_reportado"] - calc, 4)),
                    get_column_letter(bmi + 1) + str(rn)
                )

            # Guardar observaciones y evaluación nutricional
            rec["observaciones_calidad"] = "; ".join(flags) or None
            records.append(rec)

**Comentario 9:** La extracción reconoce el bloque «curva (percentiles OMS)» y el bloque «tabla (diagnóstico rápido INS)» de cada hoja, distingue cuatro estados de cada diagnóstico (categoría registrada, asterisco, celda vacía, indicador ausente en la plantilla), conserva las fórmulas de edad con su último resultado guardado (sin ejecutar `TODAY` a la fecha actual) y compara cada evaluación con la ficha nominal.

In [33]:
nutrition_df = pd.DataFrame(records)
IMC_REVIEW_TOLERANCE = 0.11
valid_measurements = (nutrition_df["peso_kg"].notna() &
                      nutrition_df["talla_cm"].notna() &
                      (nutrition_df["talla_cm"] > 0))
nutrition_df.loc[valid_measurements, "imc_calculado"] = (
    nutrition_df.loc[valid_measurements, "peso_kg"] /
    (nutrition_df.loc[valid_measurements, "talla_cm"] / 100) ** 2
)
nutrition_df["imc_calculado"] = pd.to_numeric(nutrition_df["imc_calculado"], errors="raise")
nutrition_df["diferencia_imc"] = nutrition_df["imc_reportado"] - nutrition_df["imc_calculado"]
nutrition_df["revision_imc"] = "no_comparable"
comparable = nutrition_df["diferencia_imc"].notna()
nutrition_df.loc[comparable, "revision_imc"] = "dentro_tolerancia"
nutrition_df.loc[comparable & (nutrition_df["diferencia_imc"].abs() > IMC_REVIEW_TOLERANCE),
                 "revision_imc"] = "revisar"
assert not nutrition_df["id_evaluacion"].duplicated().any()
# Mantener el mismo orden de columnas que el conjunto consolidado.
nutrition_columns = ['id_evaluacion', 'id_persona', 'periodo', 'grupo_hogar', 'casa', 'peso_kg', 'talla_cm', 'imc_reportado', 'imc_calculado', 'diferencia_imc', 'revision_imc', 'dx_pe_curva', 'dx_te_curva', 'dx_pt_curva', 'dx_imc_curva', 'dx_pe_tabla', 'dx_te_tabla', 'dx_pt_tabla', 'dx_imc_tabla', 'estado_mediciones', 'fecha_nacimiento', 'fecha_evaluacion', 'edad_reportada', 'edad_formula_guardada', 'sexo_nominal', 'estado_vinculo_nominal', 'metodo_curva_original', 'metodo_tabla_original', 'indicador_imc_curva_original', 'indicador_imc_tabla_original', 'peso_original', 'talla_original', 'imc_original', 'imc_formula_original', 'perimetro_cefalico_original', 'perimetro_cefalico_unidad', 'pa_original', 'pa_encabezado_original', 'pa_unidad', 'fecha_nacimiento_original', 'edad_formula_original', 'numero_original', 'observacion_original', 'archivo_origen', 'hoja_origen', 'fila_origen', 'columnas_diagnostico_originales', 'sexo_archivo_origen', 'sexo_hoja_origen', 'sexo_fila_origen', 'fecha_nacimiento_nominal', 'coincidencias_nominal_mes', 'identidad_nominal_observacion', 'asteriscos_diagnostico', 'vacios_diagnostico', 'indicadores_ausentes', 'normalizaciones_texto', 'diferencias_entre_metodos', 'observaciones_calidad', 'dx_pe_curva_original', 'dx_pe_curva_estado', 'dx_te_curva_original', 'dx_te_curva_estado', 'dx_pt_curva_original', 'dx_pt_curva_estado', 'dx_imc_curva_original', 'dx_imc_curva_estado', 'dx_pe_tabla_original', 'dx_pe_tabla_estado', 'dx_te_tabla_original', 'dx_te_tabla_estado', 'dx_pt_tabla_original', 'dx_pt_tabla_estado', 'dx_imc_tabla_original', 'dx_imc_tabla_estado']
nutrition_df = nutrition_df.reindex(columns=nutrition_columns)
normalization_summary = pd.DataFrame([
    {"original": original, "normalizado": normalized, "frecuencia": count}
    for (original, normalized), count in normalization_counts.items()
]).sort_values("frecuencia", ascending=False)
sheet_notes = pd.DataFrame(notes)
quality_issues = pd.DataFrame(issues)
show_table(pd.crosstab(nutrition_df["periodo"], nutrition_df["grupo_hogar"]),
           "Evaluaciones extraídas por mes y hogar")
show_table(nutrition_df["estado_mediciones"].value_counts().rename("evaluaciones"),
           "Disponibilidad de mediciones")
show_table(normalization_summary[normalization_summary["original"] != normalization_summary["normalizado"]],
           "Cambios de texto observados en los diagnósticos")
show_table(nutrition_df["revision_imc"].value_counts().rename("evaluaciones"),
           "Revisión aritmética del IMC")

print("Evaluaciones extraídas:", len(nutrition_df))

grupo_hogar,INFANTES,MATOGROSO HOMBRES,MATOGROSO MUJERES,MIGUEL ECHENIQUE,TOMAS VALLE
periodo,,,,,
2026-01,33,37,32,17,22
2026-02,7,0,0,0,0
2026-03,6,0,0,0,0
2026-04,30,24,18,16,20
2026-06,30,22,18,17,21
2026-08,0,17,15,0,0


,evaluaciones
estado_mediciones,
peso_talla_registrados,379
sin_medicion_provincia,23


,original,normalizado,frecuencia
16,NORMAL,NORMAL,305
4,RIESGO TALLA BAJA,RIESGO TALLA BAJA,98
43,Normal,NORMAL,88
15,RIESGO SOBREPESO,RIESGO SOBREPESO,75
18,SOBREPESO,SOBREPESO,28
14,TALLA BAJA,TALLA BAJA,25
7,RIESGO SOBREPESO,RIESGO SOBREPESO,15
17,OBESIDAD,OBESIDAD,11
45,Riesgo de \ntalla baja,RIESGO TALLA BAJA,10
22,ALTO,ALTO,10


,evaluaciones
revision_imc,
dentro_tolerancia,364
no_comparable,29
revisar,9


Evaluaciones extraídas: 402


**Comentario 10:**

- **Cobertura:** enero concentra más evaluaciones (141); febrero y marzo solo tienen infantes; agosto solo los hogares Matogroso.
- **Mediciones:** **379** evaluaciones traen peso y talla; **23** traen el texto `PROVINCIA` en lugar del peso (se analiza en la sección 9).
- **Texto de diagnósticos:** la tabla de cambios muestra que casi todos los «cambios» son espacios sobrantes (por ejemplo `NORMAL ` → `NORMAL`, 305 veces), seguidos de diferencias de mayúsculas (`Normal`, 88) y de errores de escritura (`RIESGO TALA BAJA`, `OBEESIDAD`, `NORNAL`…). Se detalla en la sección 6.
- **IMC:** al comparar el IMC reportado con peso/talla² con una tolerancia de 0.11, 364 evaluaciones están dentro de la tolerancia, 29 no son comparables porque falta algún dato para poder calcular el IMC y 9 difieren más del límite, es decir que el valor que se calculó con el que figura en excel difieren en mas de 0.11.

# 6. Formato del texto en los diagnósticos nutricionales

Los diagnósticos (P/E, T/E, P/T e IMC, en el bloque OMS y en el bloque INS) se escribieron a mano, por lo que la misma categoría aparece escrita de varias maneras. Si no se unifica, un conteo por categoría repartiría un mismo diagnóstico en varias filas.

In [36]:
# FORMATO DEL TEXTO EN LOS DIAGNÓSTICOS NUTRICIONALES
# Se toman los 8 bloques de diagnóstico (P/E, T/E, P/T e IMC, en curva OMS y en tabla INS) tal como llegaron
# (columnas *_original) y se comparan contra su versión normalizada (clean_text + mayúsculas + correcciones explícitas).

identificadores = ["id_evaluacion", "id_persona", "periodo", "archivo_origen", "hoja_origen", "fila_origen"]
columnas_dx_original = [c for c in nutrition_df.columns if c.startswith("dx_") and c.endswith("_original")]

dx_largo = nutrition_df[identificadores + columnas_dx_original].melt(
    id_vars=identificadores, var_name="columna_dx", value_name="valor_original")
dx_largo["columna_dx"] = dx_largo["columna_dx"].str.replace("_original", "", regex=False)
dx_largo = dx_largo[dx_largo["valor_original"].map(lambda v: isinstance(v, str) and clean_text(v) not in ("", "*"))].copy()
dx_largo["valor_normalizado"] = dx_largo["valor_original"].map(normalize_diagnosis)

def tipo_cambio(original, normalizado):
    """Clasifica por qué difiere el texto original del normalizado."""
    visible = clean_text(original)
    if original == normalizado: return "sin_cambio"
    if visible == normalizado: return "solo_espacios_o_saltos_de_linea"
    if visible.upper() == normalizado: return "mayusculas_minusculas"
    return "variante_o_error_tipografico"

dx_largo["tipo_cambio"] = [tipo_cambio(o, n) for o, n in zip(dx_largo["valor_original"], dx_largo["valor_normalizado"])]

show_table(dx_largo["tipo_cambio"].value_counts().rename("celdas"), "Celdas de diagnóstico según el tipo de diferencia de texto")

variantes_dx = (dx_largo[dx_largo["tipo_cambio"] != "sin_cambio"]
                .assign(valor_original_visible=lambda d: d["valor_original"].map(repr))
                .groupby(["valor_original_visible", "valor_normalizado", "tipo_cambio"]).size()
                .rename("frecuencia").reset_index().sort_values("frecuencia", ascending=False))
show_table(variantes_dx.set_index("valor_original_visible"), "Variantes de texto detectadas y su forma normalizada")

# Categorías que quedan después de normalizar, por indicador y bloque
categorias_dx = (dx_largo.assign(indicador=dx_largo["columna_dx"].str.extract(r"dx_(\w+?)_(?:curva|tabla)")[0],
                                 bloque=dx_largo["columna_dx"].str.extract(r"_(curva|tabla)$")[0])
                 .groupby(["indicador", "bloque", "valor_normalizado"]).size().rename("frecuencia").reset_index())
show_table(categorias_dx.set_index(["indicador", "bloque"]), "Categorías resultantes por indicador y bloque")

export_evidence("05_texto_diagnosticos.xlsx", {
    "resumen_por_tipo_de_cambio": dx_largo["tipo_cambio"].value_counts().rename("celdas").reset_index(),
    "variantes_y_normalizacion": variantes_dx,
    "categorias_resultantes": categorias_dx,
    "celdas_con_cambio_detalle": dx_largo[dx_largo["tipo_cambio"] != "sin_cambio"]})

,celdas
tipo_cambio,
sin_cambio,1003
solo_espacios_o_saltos_de_linea,596
mayusculas_minusculas,103
variante_o_error_tipografico,46


,valor_normalizado,tipo_cambio,frecuencia
valor_original_visible,,,
'NORMAL ',NORMAL,solo_espacios_o_saltos_de_linea,305
'RIESGO TALLA BAJA ',RIESGO TALLA BAJA,solo_espacios_o_saltos_de_linea,98
'Normal',NORMAL,mayusculas_minusculas,88
'RIESGO SOBREPESO ',RIESGO SOBREPESO,solo_espacios_o_saltos_de_linea,75
'SOBREPESO ',SOBREPESO,solo_espacios_o_saltos_de_linea,28
'TALLA BAJA ',TALLA BAJA,solo_espacios_o_saltos_de_linea,25
'RIESGO SOBREPESO ',RIESGO SOBREPESO,solo_espacios_o_saltos_de_linea,15
'OBESIDAD ',OBESIDAD,solo_espacios_o_saltos_de_linea,11
'Riesgo de \ntalla baja',RIESGO TALLA BAJA,variante_o_error_tipografico,10


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/05_texto_diagnosticos.xlsx  (resumen_por_tipo_de_cambio: 4 filas, variantes_y_normalizacion: 36 filas, categorias_resultantes: 46 filas, celdas_con_cambio_detalle: 745 filas)


**Comentario 11:** De 1 748 celdas con una categoría escrita, 1 003 ya estaban limpias y 745 (43 %) difieren de su forma normalizada:

- **596** solo tienen espacios o saltos de línea sobrantes (`NORMAL `, `RIESGO TALLA BAJA `, `Riesgo de \nSobrepeso`).
- **103** difieren en mayúsculas/minúsculas (`Normal`, `Sobrepeso`, `Talla baja`, `NORMAl`).
- **46** son variantes o errores tipográficos: `RIESGO DE SOBREPESO`, `RIESGO SOBRE PESO`, `RIESGO TALA BAJA`, `RIESO TALLA BAJA`, `OBEESIDAD`, `OBESIDA`, `NORNAL`, `Riego de delgadez…`.

**Categorías que quedan después de normalizar:** además de las esperables (`NORMAL`, `RIESGO SOBREPESO`, `SOBREPESO`, `OBESIDAD`, `TALLA BAJA`, `RIESGO TALLA BAJA`, `TALLA BAJA SEVERA`), persisten dos tipos de diferencia: (1) `ALTA` (2 celdas) frente a `ALTO` (16), que es la misma categoría con distinto género, (2) vocabularios distintos entre bloques (`BAJO PESO`, `RIESGO BAJO PESO`, `DESNUTRIDO`, `OBESIDAD III`, `RIESGO OBESIDAD`), que son categorías diferentes y no errores de escritura, por lo que se dejan tal cual para el análisis posterior.

# 7. Formato y consistencia de la edad

Las tablas de referencia INS se consultan por edad, así que hay que saber si la edad escrita en los Excel es confiable.

In [37]:
def parse_age_months(value):
    # Convierte la edad escrita a meses totales. Ej.: '7 años 3 meses' -> 87.
    # Acepta las variantes observadas: año/años, mes/meses/mese, 'y', mayúsculas y espacios extra.
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    texto = clean_text(value).lower()
    anios = re.search(r"(\d+)\s*a[ñn]os?", texto)
    meses = re.search(r"(\d+)\s*mes", texto)
    if not anios and not meses:
        return None
    return (int(anios.group(1)) if anios else 0) * 12 + (int(meses.group(1)) if meses else 0)

# La edad llega como texto. Si la celda era una fórmula (TODAY), se usa el último resultado guardado.
edades = nutrition_df[identificadores + ["fecha_nacimiento", "edad_reportada", "edad_formula_guardada"]].copy()
edades["edad_texto"] = edades["edad_reportada"].where(edades["edad_reportada"].notna(), edades["edad_formula_guardada"])
edades["edad_meses"] = edades["edad_texto"].map(parse_age_months)
edades["molde_de_escritura"] = edades["edad_texto"].map(lambda v: re.sub(r"\d+", "N", str(v)))

show_table(edades["molde_de_escritura"].value_counts().rename("evaluaciones"), "Formas en que viene escrita la edad")
print("Edades que no se pudieron convertir a meses:", int(edades["edad_meses"].isna().sum()))

# Contraste con la fecha de nacimiento: meses entre nacimiento y el mes de la evaluación
periodo_meses = edades["periodo"].map(lambda p: int(p[:4]) * 12 + int(p[5:7]))
nacimiento_meses = pd.to_datetime(edades["fecha_nacimiento"]).map(lambda d: d.year * 12 + d.month)
edades["edad_meses_segun_nacimiento"] = periodo_meses - nacimiento_meses
edades["diferencia_meses"] = edades["edad_meses"] - edades["edad_meses_segun_nacimiento"]
show_table(edades["diferencia_meses"].describe().to_frame("diferencia (meses)"),
           "Edad escrita menos edad según fecha de nacimiento")
show_table(edades.groupby("periodo")["diferencia_meses"].agg(["count", "mean", "min", "max"]).round(2),
           "Edad escrita menos edad según nacimiento, por mes de evaluación")
edades_dudosas = edades[edades["diferencia_meses"].abs() > 2]
print("Evaluaciones con diferencia > 2 meses:", len(edades_dudosas))

export_evidence("06_edades_formato_y_consistencia.xlsx", {
    "moldes_de_escritura": edades["molde_de_escritura"].value_counts().rename("evaluaciones").reset_index(),
    "edad_no_convertible": edades[edades["edad_meses"].isna()],
    "edad_vs_nacimiento_dudosa": edades_dudosas})

,evaluaciones
molde_de_escritura,
N años N meses,164
N años N meses,116
N años,22
N años y N Meses,17
N años N mes,16
N años y Nmeses,15
N año N meses,9
N años,7
N años N mese,6


Edades que no se pudieron convertir a meses: 0


,diferencia (meses)
count,402.000000
mean,0.890547
std,2.236171
min,-12.000000
25%,0.000000
50%,0.000000
75%,3.000000
max,12.000000


,count,mean,min,max
periodo,,,,
2026-01,141,0.21,-12,12
2026-02,7,-1.43,-12,1
2026-03,6,0.33,0,1
2026-04,108,-0.49,-3,3
2026-06,108,3.86,3,4
2026-08,32,-0.88,-1,0


Evaluaciones con diferencia > 2 meses: 120
Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/06_edades_formato_y_consistencia.xlsx  (moldes_de_escritura: 23 filas, edad_no_convertible: 0 filas, edad_vs_nacimiento_dudosa: 120 filas)


**Comentario 12:**

- **Formato:** hay 23 variantes de escritura (`7 años 3 meses`, `7 año 3 mes`, `16 años y 1meses`, `7 años`, `2 meses`, `14 años 10 años`…), con diferencias de singular/plural, mayúsculas y espacios. Con el convertidor a meses las 402 se pudieron interpretar, aunque una (`14 años 10 años`) es un error de digitación cuya parte de meses se pierde.
- **Consistencia con la fecha de nacimiento:** al restar la fecha de nacimiento al mes de la evaluación, 120 evaluaciones difieren en más de 2 meses. El patrón es muy claro, las 108 evaluaciones de junio están 3 o 4 meses adelantadas (diferencia media de +3.3 meses), mientras que en los demás meses la diferencia es casi siempre 0. Esto indica que la edad de junio venía de una fórmula con `TODAY` y se recalculó en una fecha posterior a la evaluación. Los demás casos son errores puntuales de digitación: 3 en enero y 1 en febrero, con diferencias de hasta 12 meses (por ejemplo, una niña de 2 años y 4 meses escrita como `1 año 4 meses`), y 8 en abril con un desfase de 3 meses.

**Consecuencia:** la edad escrita en el Excel no debe usarse para decidir qué indicador o fila de tabla corresponde. En el Paso 3 la edad se calcula desde la fecha de nacimiento y el mes de la evaluación, y la edad escrita queda solo como contraste.

# 8. Asteriscos (`*`) en los diagnósticos

Un `*` en una columna de diagnóstico significa que ese valor **no se calculó**. Según la regla de negocio, los menores de 5 años usan **P/E, T/E y P/T**, y de 5 años a más solo **T/E e IMC**. Por eso un asterisco puede ser legítimo («no corresponde a esa edad») o ser un valor que falta calcular. Se clasifica cada uno por su motivo.

In [38]:
# ASTERISCOS (*) EN LOS DIAGNÓSTICOS: ¿CUÁLES CORRESPONDEN A UN VALOR NO CALCULADO?
# Regla de negocio: menores de 5 años -> P/E, T/E y P/T; de 5 años a más -> T/E e IMC.
# Un '*' en un indicador que NO aplica por edad es legítimo ("no corresponde"); en uno que sí aplica, falta calcularlo.
APLICA_POR_EDAD = {"menor_5": {"pe", "te", "pt"}, "desde_5": {"te", "imc"}}

def grupo_etario(meses):
    if meses is None or pd.isna(meses): return "sin_edad"
    return "menor_5" if meses < 60 else "desde_5"

# Edad usada: meses entre el nacimiento y el mes de la evaluación (la edad escrita en junio viene adelantada, ver sección anterior)
base_ast = nutrition_df.merge(edades[["id_evaluacion", "edad_meses_segun_nacimiento"]].rename(
    columns={"edad_meses_segun_nacimiento": "edad_meses"}), on="id_evaluacion", how="left")
filas_ast = []
for indicador in ("pe", "te", "pt", "imc"):
    for bloque in ("curva", "tabla"):
        marcados = base_ast[base_ast[f"dx_{indicador}_{bloque}_estado"] == "asterisco_original"]
        for _, r in marcados.iterrows():
            grupo = grupo_etario(r["edad_meses"])
            necesita_peso = indicador in ("pe", "pt", "imc")
            necesita_talla = indicador in ("te", "pt", "imc")
            if grupo == "sin_edad":
                motivo = "sin_edad_para_decidir"
            elif indicador not in APLICA_POR_EDAD[grupo]:
                motivo = "no_aplica_por_edad"
            elif r["edad_meses"] >= 240:
                motivo = "edad_fuera_del_rango_5_a_19_anios"
            elif (necesita_peso and pd.isna(r["peso_kg"])) or (necesita_talla and pd.isna(r["talla_cm"])):
                motivo = "aplica_pero_faltan_mediciones"
            elif bloque == "curva":
                motivo = "aplica_pero_bloque_OMS_sin_tabla_numerica"
            else:
                motivo = "aplica_y_calculable_con_tablas_INS"
            filas_ast.append({
                "id_evaluacion": r["id_evaluacion"], "id_persona": r["id_persona"], "periodo": r["periodo"],
                "indicador": indicador, "bloque": bloque, "grupo_etario": grupo, "edad_meses": r["edad_meses"],
                "peso_kg": r["peso_kg"], "talla_cm": r["talla_cm"], "motivo": motivo,
                "archivo_origen": r["archivo_origen"], "hoja_origen": r["hoja_origen"], "fila_origen": r["fila_origen"]})
asteriscos = pd.DataFrame(filas_ast)

resumen_ast = pd.crosstab([asteriscos["indicador"], asteriscos["bloque"]], asteriscos["motivo"], margins=True, margins_name="TOTAL")
show_table(resumen_ast, "Asteriscos por indicador, bloque y motivo")
show_table(pd.crosstab(asteriscos["indicador"], asteriscos["grupo_etario"]), "Asteriscos por indicador y grupo etario")

export_evidence("07_asteriscos_diagnostico.xlsx", {
    "resumen": resumen_ast.reset_index(),
    **{motivo[:31]: grupo for motivo, grupo in asteriscos.groupby("motivo")}})

grupo_etario,desde_5,menor_5
indicador,,
imc,2,76
pe,135,2
pt,76,11
te,4,0


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/07_asteriscos_diagnostico.xlsx  (resumen: 9 filas, aplica_pero_bloque_OMS_sin_tabl: 11 filas, aplica_y_calculable_con_tablas_: 3 filas, edad_fuera_del_rango_5_a_19_ani: 5 filas, no_aplica_por_edad: 287 filas)


**Comentario 13:** Hay 306 asteriscos:

| Motivo | Cantidad | Qué significa |
|---|---|---|
| No aplica por edad | **287** | Por ejemplo, P/E y P/T en mayores de 5 años, o IMC en menores de 5. El `*` es correcto. |
| Aplica pero es del bloque OMS | **11** | P/T en el bloque «percentiles OMS» de niños de 2 a 4 años. El PDF de OMS entregado solo contiene gráficos de percentiles, sin valores numéricos, así que no se puede calcular con exactitud. |
| Edad fuera del rango de las tablas | **5** | Todos del código **PPA_FN_0054**, una persona de alrededor de 28 años según la edad escrita (la tabla INS cubre hasta los 19 años y 11 meses). |
| **Aplica y es calculable con tablas INS** | **3** | P/E de dos niños de 4 años y 7 meses (junio) y T/E de una adolescente de 18 años (enero). |

**Conclusión:** de los 306 asteriscos solo **3** son un valor realmente faltante que se puede calcular con las tablas INS. Los demás son «no corresponde» o no se pueden calcular con la documentación disponible. También se observa que PPA_FN_0054 es un adulto dentro de los hogares de escolares y adolescentes.

# 9. Evaluaciones con la palabra `PROVINCIA`

En algunas filas la celda de peso contiene el texto `PROVINCIA` en lugar de un número. Se revisa dónde aparece, a quiénes afecta y si esas filas traen algún otro dato.

In [39]:
# EVALUACIONES CON LA PALABRA "PROVINCIA" EN LUGAR DE MEDICIONES
provincia = nutrition_df[nutrition_df["estado_mediciones"] == "sin_medicion_provincia"].copy()

# ¿En qué columnas de la fila aparece la palabra? (se revisan todas las columnas de texto original)
columnas_con_provincia = {}
for columna in ["peso_original", "talla_original", "imc_original", "observacion_original"]:
    columnas_con_provincia[columna] = int(provincia[columna].astype(str).str.upper().str.contains("PROVINCIA").sum())
show_table(pd.Series(columnas_con_provincia, name="filas con la palabra PROVINCIA"), "Dónde aparece la palabra")

show_table(pd.crosstab(provincia["periodo"], provincia["grupo_hogar"], margins=True, margins_name="TOTAL"),
           "Evaluaciones con PROVINCIA por mes y hogar")
# ¿Esas filas traen diagnósticos pese a no tener mediciones?
con_dx = provincia[[c for c in provincia.columns if c.startswith("dx_") and c.endswith("_estado")]].eq("categoria_registrada").any(axis=1)
print("Filas PROVINCIA que aun así traen algún diagnóstico registrado:", int(con_dx.sum()), "de", len(provincia))
show_table(provincia[["id_persona", "periodo", "casa", "peso_original", "talla_original", "archivo_origen", "hoja_origen", "fila_origen"]]
           .set_index("id_persona"), "Evaluaciones con PROVINCIA")

export_evidence("08_evaluaciones_provincia.xlsx", {"evaluaciones_provincia": provincia})

,filas con la palabra PROVINCIA
peso_original,23
talla_original,0
imc_original,0
observacion_original,23


grupo_hogar,MATOGROSO HOMBRES,MATOGROSO MUJERES,TOTAL
periodo,,,
2026-01,5,4,9
2026-04,4,3,7
2026-06,4,3,7
TOTAL,13,10,23


Filas PROVINCIA que aun así traen algún diagnóstico registrado: 0 de 23


,periodo,casa,peso_original,talla_original,archivo_origen,hoja_origen,fila_origen
id_persona,,,,,,,
PPA_FN_0027,2026-01,CASA ADOLESCENTES MUJERES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,25
PPA_FN_0044,2026-01,CASA ADOLESCENTES MUJERES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,30
PPA_FN_0045,2026-01,CASA ADOLESCENTES MUJERES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,31
PPA_FN_0062,2026-01,CASA ADOLESCENTES MUJERES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,37
PPA_FN_0072,2026-01,CASA ADOLESCENTES VARONES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,32
PPA_FN_0083,2026-01,CASA ADOLESCENTES VARONES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,37
PPA_FN_0088,2026-01,CASA ADOLESCENTES VARONES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,39
PPA_FN_0093,2026-01,CASA ADOLESCENTES VARONES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,40
PPA_FN_0096,2026-01,CASA ADOLESCENTES VARONES PROVINCIA,PROVINCIA,—,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,42


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/08_evaluaciones_provincia.xlsx  (evaluaciones_provincia: 23 filas)


**Comentario 14:** La palabra aparece solo en la columna de peso (23 filas), la talla y el IMC quedan vacíos y ninguna de estas filas trae diagnóstico registrado. Afecta únicamente a los hogares Matogroso que son 13 varones y 10 mujeres, en enero (9), abril (7) y junio (7). Son las mismas personas mes a mes (por ejemplo PPA_FN_0043, 0137, 0061, 0068, 0069, 0077), lo que sugiere adolescentes que residen temporalmente en provincia y no fueron evaluados.

**Interpretación y tratamiento:** `PROVINCIA` no es un peso cero ni un valor perdido por error; indica que en ese mes esa persona no tuvo evaluación. El código original ya las clasifica como «sin medición provincia» y no las convierte en cero, el Paso 3 se debería excluir de la nutrición (la ficha nominal de ese mes se conserva, marcada como «sin evaluación por PROVINCIA»).

En agosto la palabra aparece en el nombre de la casa (`CASA ADOLESCENTES … PROVINCIA`), pero esas filas sí traen peso y talla numéricos, por lo que no se consideran evaluaciones omitidas.

# 10. IMC: faltantes y diferencias contra el cálculo

El IMC se compara con peso (kg) / talla (m)². Se separan dos situaciones: IMC que no se puede comparar y IMC reportado que difiere del calculado.

In [40]:
# Evaluaciones con IMC no comparable
no_comparable = nutrition_df[
    nutrition_df["revision_imc"] == "no_comparable"
]

display(no_comparable[[
    "id_persona",
    "periodo",
    "archivo_origen",
    "hoja_origen",
    "fila_origen"
]])

export_evidence("09_imc_no_comparable.xlsx", {"imc_no_comparable": nutrition_df[nutrition_df["revision_imc"] == "no_comparable"]})

,id_persona,periodo,archivo_origen,hoja_origen,fila_origen
90,PPA_FN_0027,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,25
95,PPA_FN_0044,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,30
96,PPA_FN_0045,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,31
102,PPA_FN_0062,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,37
129,PPA_FN_0072,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,32
134,PPA_FN_0083,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,37
136,PPA_FN_0088,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,39
137,PPA_FN_0093,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,40
139,PPA_FN_0096,2026-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,42
148,PPA_FN_0117,2026-03,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,5


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/09_imc_no_comparable.xlsx  (imc_no_comparable: 29 filas)


De los indices que aparecen, hay evaluaciones en las que el IMC no se reporta o tiene un texto (NORMAL), por eso se van a seleccionar cuales son para que en un posterior tratamiento se pueda imputar con los IMC correctos.

In [41]:
# Seleccionar evaluaciones con IMC no comparable,
# pero con peso y talla disponibles para calcularlo

imc_imputable = nutrition_df[
    (nutrition_df["revision_imc"] == "no_comparable") &
    (nutrition_df["peso_kg"].notna()) &
    (nutrition_df["talla_cm"].notna()) &
    (nutrition_df["talla_cm"] > 0) &
    (nutrition_df["imc_calculado"].notna())
].copy()

# Mostrar los valores y su ubicación en el Excel original
display(imc_imputable[[
    "id_persona",
    "periodo",
    "peso_kg",
    "talla_cm",
    "imc_original",
    "imc_reportado",
    "imc_calculado",
    "archivo_origen",
    "hoja_origen",
    "fila_origen"
]])

print("Registros candidatos para completar el IMC:", len(imc_imputable))

export_evidence("10_imc_imputable.xlsx", {"imc_imputable_con_peso_y_talla": imc_imputable})

,id_persona,periodo,peso_kg,talla_cm,imc_original,imc_reportado,imc_calculado,archivo_origen,hoja_origen,fila_origen
148,PPA_FN_0117,2026-03,14.04,92.5,NaN,NaN,16.409058,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,5
149,PPA_FN_0140,2026-03,14.30,85.0,NaN,NaN,19.792388,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,6
150,PPA_FN_0157,2026-03,11.30,77.0,NaN,NaN,19.058863,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,7
151,PPA_FN_0153,2026-03,10.40,73.5,NaN,NaN,19.251238,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,8
152,PPA_FN_0146,2026-03,15.40,91.0,NaN,NaN,18.596788,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,9
153,PPA_FN_0149,2026-03,10.50,73.5,NaN,NaN,19.436346,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,10


Registros candidatos para completar el IMC: 6
Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/10_imc_imputable.xlsx  (imc_imputable_con_peso_y_talla: 6 filas)


In [42]:
# Evaluaciones con diferencias en el IMC
revisar = nutrition_df[
    nutrition_df["revision_imc"] == "revisar"
]

display(revisar[[
    "id_persona",
    "periodo",
    "archivo_origen",
    "hoja_origen",
    "fila_origen"
]])

export_evidence("11_imc_diferencias_mayores.xlsx", {"imc_reportado_vs_calculado": revisar})

,id_persona,periodo,archivo_origen,hoja_origen,fila_origen
370,PPA_FN_0027,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO MUJERES,5
371,PPA_FN_0032,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO MUJERES,6
374,PPA_FN_0045,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO MUJERES,9
384,PPA_FN_0044,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO MUJERES,19
386,PPA_FN_0067,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,6
394,PPA_FN_0080,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,14
398,PPA_FN_0095,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,18
400,PPA_FN_0099,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,20
401,PPA_FN_0072,2026-08,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,21


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/11_imc_diferencias_mayores.xlsx  (imc_reportado_vs_calculado: 9 filas)


**Comentario 15:**

1. 29 evaluaciones no comparables. 23 son las filas `PROVINCIA` (sin peso ni talla, por lo que no hay nada que calcular). Las otras 6 son de marzo(PPA_FN_0117, 0140, 0157, 0153, 0146 y 0149): tienen peso y talla válidos, pero el IMC no fue reportado porque la plantilla de marzo reemplazó esa columna por el perímetro cefálico. Estas 6 sí se pueden completar con la fórmula (IMC entre 16.4 y 19.8).
2. 9 evaluaciones con diferencia mayor a 0.11. Todas son de agosto (PPA_FN_0027, 0032, 0045, 0044, 0067, 0080, 0095, 0099 y 0072), con diferencias de entre −0.20 y +0.46 puntos de IMC. Como el IMC de agosto fue digitado con uno o dos decimales, el redondeo explicaría como máximo unas centésimas, así que estas diferencias son demasiado grandes para ser redondeo y apuntan a un peso o talla digitado distinto del usado para el IMC. No se puede decidir cuál es el correcto, por lo que se conserva el IMC reportado y se deja señalado para revisión.

# 11. Consistencia entre las dos fuentes

Una evaluación solo se une a una ficha nominal si coinciden el **código y el mes**, hay **una única ficha candidata**, coincide la **fecha de nacimiento** y el código **no tiene alertas de identidad**. Aquí se mide cuántas evaluaciones cumplen y por qué las demás no.

In [43]:
# ¿CUÁNTAS EVALUACIONES SE PUEDEN VINCULAR CON SEGURIDAD A LA FICHA NOMINAL?
# El estado de vínculo ya se calculó en la extracción (periodo + identificador + fecha de nacimiento + código sin alertas).
estado_vinculo = nutrition_df["estado_vinculo_nominal"].value_counts().rename("evaluaciones")
show_table(estado_vinculo, "Estado de vínculo de las evaluaciones con la ficha nominal")

en_ids_inconsistentes = nutrition_df[nutrition_df["id_persona"].isin(conflicting_ids)]
print("Evaluaciones de códigos inconsistentes:", len(en_ids_inconsistentes))
show_table(en_ids_inconsistentes["id_persona"].value_counts().rename("evaluaciones"), "Evaluaciones por código inconsistente")

# Cada estado se guarda en su propia hoja para poder revisarlos por separado
hojas_vinculo = {"resumen": estado_vinculo.reset_index()}
for estado, grupo in nutrition_df.groupby("estado_vinculo_nominal"):
    hojas_vinculo[estado[:31]] = grupo[["id_evaluacion", "id_persona", "periodo", "fecha_nacimiento",
                                         "fecha_nacimiento_nominal", "coincidencias_nominal_mes",
                                         "identidad_nominal_observacion", "archivo_origen", "hoja_origen", "fila_origen"]]
export_evidence("12_vinculo_nutricion_nominal.xlsx", hojas_vinculo)
export_evidence("13_evaluaciones_de_ids_inconsistentes.xlsx", {"evaluaciones_nutricion": en_ids_inconsistentes})

,evaluaciones
estado_vinculo_nominal,
unico_mes_y_nacimiento_coinciden,331
periodo_nominal_no_disponible,32
identidad_contradictoria_no_vinculada,24
fecha_nacimiento_no_coincide,10
id_ausente_en_mes_nominal,5


Evaluaciones de códigos inconsistentes: 24


,evaluaciones
id_persona,
PPA_FN_0127,3
PPA_FN_0010,3
PPA_FN_0036,3
PPA_FN_0035,3
PPA_FN_0082,3
PPA_FN_0071,3
PPA_FN_0020,3
PPA_FN_0025,3


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/12_vinculo_nutricion_nominal.xlsx  (resumen: 5 filas, fecha_nacimiento_no_coincide: 10 filas, id_ausente_en_mes_nominal: 5 filas, identidad_contradictoria_no_vin: 24 filas, periodo_nominal_no_disponible: 32 filas, unico_mes_y_nacimiento_coincide: 331 filas)
Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/13_evaluaciones_de_ids_inconsistentes.xlsx  (evaluaciones_nutricion: 24 filas)


**Comentario 16:**

| Estado | Evaluaciones | Causa |
|---|---|---|
| Vínculo seguro | **331** | Código, mes y nacimiento coinciden. |
| Periodo nominal no disponible | **32** | Todas de agosto: la ficha nominal termina en 2026-07. |
| Identidad contradictoria | **24** | Los 8 códigos inconsistentes (3 evaluaciones cada uno). |
| Fecha de nacimiento no coincide | **10** | PPA_FN_0004, 0028 y 0066 (enero, abril y junio) y PPA_FN_0064 (enero). |
| Código ausente en el mes nominal | **5** | PPA_FN_0144 (enero y febrero), 0155, 0154 y 0090 (enero). |

Las diferencias de nacimiento muestran el problema de formato de fechas de la sección 2: PPA_FN_0004 (`2023-01-09` frente a `2023-09-01`) y PPA_FN_0066 (`2016-09-02` frente a `2016-02-09`) tienen día y mes intercambiados, mientras que 0028 y 0064 difieren por 1 a 4 días. Los códigos ausentes en el mes nominal corresponden a niños que figuran en nutrición pero no en la ficha de ese mes. Todos estos casos se guardan en hojas separadas del Excel `12_vinculo_nutricion_nominal.xlsx` para que se corrijan en la fuente.

# 12. Cobertura del sexo

Las tablas INS son distintas para mujeres y varones, de modo que, para calcular un diagnóstico, hay que conocer el sexo de cada NNA. Los libros de nutrición no lo traen, entonces se toma de la ficha nominal.

In [44]:
# COBERTURA DEL SEXO: las tablas INS son distintas para mujeres y varones, por lo que se necesita el sexo de cada NNA.
# Se busca en la ficha nominal de CUALQUIER mes (el sexo de una persona no cambia entre meses).
sexos_por_id = revision.groupby("id_persona")["sexo"].agg(lambda s: sorted(set(s)))

def cobertura_sexo(ident):
    if ident not in sexos_por_id.index: return "sin_ficha_nominal"
    return "un_solo_sexo" if len(sexos_por_id[ident]) == 1 else "sexo_contradictorio"

cobertura = nutrition_df["id_persona"].map(cobertura_sexo)
show_table(pd.crosstab(cobertura, nutrition_df["periodo"], margins=True, margins_name="TOTAL"),
           "Evaluaciones según cobertura del sexo en la ficha nominal")

sin_sexo_usable = nutrition_df[cobertura != "un_solo_sexo"][["id_evaluacion", "id_persona", "periodo"]].assign(cobertura=cobertura)
export_evidence("14_cobertura_sexo.xlsx", {
    "resumen": cobertura.value_counts().rename("evaluaciones").reset_index(),
    "evaluaciones_sin_sexo_usable": sin_sexo_usable})

periodo,2026-01,2026-02,2026-03,2026-04,2026-06,2026-08,TOTAL
id_persona,,,,,,,
sexo_contradictorio,1,0,0,1,1,0,3
un_solo_sexo,140,7,6,107,107,32,399
TOTAL,141,7,6,108,108,32,402


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/14_cobertura_sexo.xlsx  (resumen: 2 filas, evaluaciones_sin_sexo_usable: 3 filas)


**Comentario 17:** Tomando el sexo de la ficha nominal de cualquier mes (el sexo de una persona no cambia), 399 de las 402 evaluaciones tienen un único sexo disponible, incluidas las 32 de agosto, que no tienen ficha nominal propia. Las 3 restantes pertenecen a PPA_FN_0020, el código con dos valores de sexo, que de todos modos queda excluido. Ninguna evaluación carece por completo de ficha nominal.

# 13. Resumen de hallazgos y tratamiento previsto

In [45]:
# RESUMEN DE HALLAZGOS Y TRATAMIENTO QUE LES CORRESPONDE EN EL PASO 3
n_cambios_texto = int((dx_largo["tipo_cambio"] != "sin_cambio").sum())
hallazgos = pd.DataFrame([
    ("Códigos (IDs) con datos contradictorios en la ficha nominal", len(conflicting_ids),
     "Excluir todas sus fichas y evaluaciones del consolidado", "03_ids_inconsistentes.xlsx / 13_evaluaciones_de_ids_inconsistentes.xlsx"),
    ("Evaluaciones nutricionales con PROVINCIA en lugar de mediciones", len(provincia),
     "Excluir: ese mes la persona no tuvo evaluación", "08_evaluaciones_provincia.xlsx"),
    ("Celdas de diagnóstico con texto mal formateado (mayúsculas, espacios, errores)", n_cambios_texto,
     "Normalizar a una sola escritura por categoría", "05_texto_diagnosticos.xlsx"),
    ("Asteriscos que aplican por edad y se pueden calcular con tablas INS", int((asteriscos["motivo"] == "aplica_y_calculable_con_tablas_INS").sum()),
     "Calcular con las tablas INS y marcar con un flag TRUE/FALSE", "07_asteriscos_diagnostico.xlsx"),
    ("Asteriscos del bloque OMS (curvas sin valores numéricos)", int((asteriscos["motivo"] == "aplica_pero_bloque_OMS_sin_tabla_numerica").sum()),
     "No se calculan; se conservan como vienen", "07_asteriscos_diagnostico.xlsx"),
    ("Asteriscos en indicadores que no aplican por edad", int((asteriscos["motivo"] == "no_aplica_por_edad").sum()),
     "Se conservan: el asterisco es legítimo", "07_asteriscos_diagnostico.xlsx"),
    ("Evaluaciones sin IMC reportado pero con peso y talla", len(imc_imputable),
     "Completar el IMC con peso/talla² y marcar con un flag", "10_imc_imputable.xlsx"),
    ("Evaluaciones con IMC reportado distinto del calculado (> 0.11)", len(revisar),
     "Se conserva el IMC reportado y se deja señalado para revisión", "11_imc_diferencias_mayores.xlsx"),
], columns=["hallazgo", "cantidad", "tratamiento_en_paso_3", "evidencia"])

show_table(hallazgos.set_index("hallazgo"), "Hallazgos del Paso 2")
export_evidence("00_resumen_hallazgos.xlsx", {"hallazgos": hallazgos})

,cantidad,tratamiento_en_paso_3,evidencia
hallazgo,,,
Códigos (IDs) con datos contradictorios en la ficha nominal,8,Excluir todas sus fichas y evaluaciones del consolidado,03_ids_inconsistentes.xlsx / 13_evaluaciones_de_ids_inconsistentes.xlsx
Evaluaciones nutricionales con PROVINCIA en lugar de mediciones,23,Excluir: ese mes la persona no tuvo evaluación,08_evaluaciones_provincia.xlsx
"Celdas de diagnóstico con texto mal formateado (mayúsculas, espacios, errores)",745,Normalizar a una sola escritura por categoría,05_texto_diagnosticos.xlsx
Asteriscos que aplican por edad y se pueden calcular con tablas INS,3,Calcular con las tablas INS y marcar con un flag TRUE/FALSE,07_asteriscos_diagnostico.xlsx
Asteriscos del bloque OMS (curvas sin valores numéricos),11,No se calculan; se conservan como vienen,07_asteriscos_diagnostico.xlsx
Asteriscos en indicadores que no aplican por edad,287,Se conservan: el asterisco es legítimo,07_asteriscos_diagnostico.xlsx
Evaluaciones sin IMC reportado pero con peso y talla,6,Completar el IMC con peso/talla² y marcar con un flag,10_imc_imputable.xlsx
Evaluaciones con IMC reportado distinto del calculado (> 0.11),9,Se conserva el IMC reportado y se deja señalado para revisión,11_imc_diferencias_mayores.xlsx


Evidencia guardada: C - KDD Steps/B - 2: Target Dataset Selection/00_resumen_hallazgos.xlsx  (hallazgos: 8 filas)


**Comentario 18:** La tabla resume cada inconsistencia, su magnitud y el tratamiento previsto.

Lo más relevante para el análisis posterior es que: (1) la identidad de 8 códigos no es confiable y se excluyen, (2) 23 evaluaciones no existen (`PROVINCIA`) y se excluyen, (3) el texto de 745 celdas de diagnóstico necesita unificarse, (4) 6 IMC se pueden completar y 9 quedan señalados para revisión, (5) solo 3 asteriscos son calculables con las tablas INS, y (6) la edad de junio está desfasada, por lo que la edad debe calcularse desde la fecha de nacimiento.